# Finetune Qwen3 with Unsloth on Dependency Analysis Data

Finetunes **Qwen3-1.7B / 4B / 8B** (Instruct) using LoRA via Unsloth on `dep_analysis_labeled.json`.

**Pipeline:**
1. Install Unsloth + pinned deps
2. Load base model (4-bit quantized for training)
3. Convert `dep_analysis_labeled.json` (ShareGPT-style) to Qwen3 chat format
4. Train with SFTTrainer (response-only loss masking)
5. Save merged 16-bit weights for vLLM inference

After training, serve the merged model:
```bash
vllm serve ./finetuned_qwen3_dep --max-model-len 16384
```
Then run `01_eval_base_models.ipynb` with `CURRENT_MODEL` pointing to the finetuned path.

## Install

Installs Unsloth with pinned `transformers` and `trl` versions (from the official Qwen3 Unsloth notebook).

In [ ]:
%%capture
import os, re
if "COLAB_" not in "".join(os.environ.keys()):
    # Local / cloud VM
    !pip install unsloth
else:
    # Google Colab — match xformers to torch version
    import torch; v = re.match(r'[\d]{1,}\.[\d]{1,}', str(torch.__version__)).group(0)
    xformers = 'xformers==' + {'2.10':'0.0.34','2.9':'0.0.33.post1','2.8':'0.0.32.post2'}.get(v, '0.0.34')
    !pip install sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer
    !pip install --no-deps unsloth_zoo bitsandbytes accelerate {xformers} peft trl triton unsloth

# Pin transformers and trl to Unsloth-tested versions
!pip install transformers==4.56.2
!pip install --no-deps trl==0.22.2

## Config

In [ ]:
from pathlib import Path

# >>>> Pick your base model <<<<
# Options: "unsloth/Qwen3-1.7B-Instruct-2507"
#          "unsloth/Qwen3-4B-Instruct-2507"
#          "unsloth/Qwen3-8B-Instruct-2507"
BASE_MODEL = "unsloth/Qwen3-4B-Instruct-2507"

# Sequence length — prompts avg ~2.2k tokens, responses ~90 tokens
MAX_SEQ_LENGTH = 4096

# Training
BATCH_SIZE = 2
GRAD_ACCUM = 4        # effective batch = 2 * 4 = 8
LEARNING_RATE = 2e-4
NUM_EPOCHS = 1        # set to 1 for full run; override with MAX_STEPS for quick test
MAX_STEPS = -1        # set to e.g. 100 for a quick sanity check; -1 = use epochs
LORA_R = 32
LORA_ALPHA = 32

# Paths
ROOT = Path(".").resolve()
LABELED_DATA = ROOT / "data" / "dep_analysis_labeled.json"
OUTPUT_DIR = ROOT / "finetuned_qwen3_dep"
LORA_DIR = ROOT / "qwen3_lora_dep"
CHECKPOINT_DIR = ROOT / "checkpoints"

print(f"Base model:  {BASE_MODEL}")
print(f"Dataset:     {LABELED_DATA}")
print(f"Output:      {OUTPUT_DIR}")

## Load model

In [ ]:
from unsloth import FastLanguageModel
import torch

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=BASE_MODEL,
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=True,
    load_in_8bit=False,
    full_finetuning=False,
)

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r=LORA_R,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    lora_alpha=LORA_ALPHA,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=3407,
)

## Prepare dataset

`dep_analysis_labeled.json` uses ShareGPT format with `from`/`value` keys.  
We convert to `role`/`content` and apply the Qwen3-Instruct chat template.

In [ ]:
import json
from datasets import Dataset
from unsloth.chat_templates import get_chat_template

tokenizer = get_chat_template(tokenizer, chat_template="qwen3-instruct")

# Load and convert from ShareGPT (from/value) to OpenAI (role/content)
with open(LABELED_DATA) as f:
    raw_data = json.load(f)

ROLE_MAP = {"human": "user", "gpt": "assistant"}

conversations = []
skipped = 0
for item in raw_data:
    if not isinstance(item, dict) or "conversations" not in item:
        skipped += 1
        continue
    convs = item["conversations"]
    if len(convs) < 2:
        skipped += 1
        continue
    converted = []
    for turn in convs:
        role = ROLE_MAP.get(turn.get("from", ""), turn.get("from", ""))
        converted.append({"role": role, "content": turn["value"]})
    conversations.append(converted)

print(f"Loaded {len(conversations)} conversations ({skipped} skipped)")

dataset = Dataset.from_dict({"conversations": conversations})
print(f"Dataset: {dataset}")

In [ ]:
# Apply chat template
def formatting_prompts_func(examples):
    texts = [
        tokenizer.apply_chat_template(
            convo, tokenize=False, add_generation_prompt=False
        )
        for convo in examples["conversations"]
    ]
    return {"text": texts}

dataset = dataset.map(formatting_prompts_func, batched=True)

# Verify
print("Sample (first 500 chars):")
print(dataset[0]["text"][:500])
print("...")

In [ ]:
# Check token length distribution
import numpy as np

sample_size = min(500, len(dataset))
lengths = [len(tokenizer.encode(dataset[i]["text"])) for i in range(sample_size)]
print(f"Token lengths (sample of {sample_size}):")
print(f"  min={min(lengths)}, median={int(np.median(lengths))}, "
      f"mean={int(np.mean(lengths))}, max={max(lengths)}")
print(f"  > {MAX_SEQ_LENGTH}: {sum(1 for l in lengths if l > MAX_SEQ_LENGTH)} samples will be truncated")

## Train

In [ ]:
from trl import SFTTrainer, SFTConfig

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    eval_dataset=None,
    args=SFTConfig(
        dataset_text_field="text",
        per_device_train_batch_size=BATCH_SIZE,
        gradient_accumulation_steps=GRAD_ACCUM,
        warmup_steps=10,
        num_train_epochs=NUM_EPOCHS,
        max_steps=MAX_STEPS,
        learning_rate=LEARNING_RATE,
        logging_steps=10,
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="cosine",
        seed=3407,
        output_dir=str(CHECKPOINT_DIR),
        save_steps=200,
        save_total_limit=3,
        report_to="none",
    ),
)

In [ ]:
# Train on assistant responses only (mask instruction tokens)
from unsloth.chat_templates import train_on_responses_only

trainer = train_on_responses_only(
    trainer,
    instruction_part="<|im_start|>user\n",
    response_part="<|im_start|>assistant\n",
)

# Verify masking
labels = trainer.train_dataset[0]["labels"]
masked = sum(1 for x in labels if x == -100)
total = len(labels)
print(f"Token masking: {masked}/{total} masked ({masked/total:.0%} instruction, {(total-masked)/total:.0%} response)")

In [ ]:
# Memory stats before training
gpu_stats = torch.cuda.get_device_properties(0)
start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
max_memory = round(gpu_stats.total_memory / 1024 / 1024 / 1024, 3)
print(f"GPU: {gpu_stats.name}, Max memory: {max_memory} GB")
print(f"Reserved before training: {start_gpu_memory} GB")

In [ ]:
trainer_stats = trainer.train()

In [ ]:
# Training summary
used_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
print(f"Training time: {trainer_stats.metrics['train_runtime']:.0f}s "
      f"({trainer_stats.metrics['train_runtime']/60:.1f} min)")
print(f"Peak GPU memory: {used_memory} GB / {max_memory} GB "
      f"({used_memory/max_memory*100:.1f}%)")
print(f"Training loss: {trainer_stats.metrics.get('train_loss', 'N/A')}")

## Quick sanity check

In [ ]:
# Test with a sample C# prompt
test_prompt = """Analyze this C# file and extract dependency information.

TASK: Extract the following:
1. NAMESPACE: What namespace is declared in this file?
2. DEFINITIONS: What classes, interfaces, enums, structs are DEFINED?
3. USINGS: What namespaces are imported? (Ignore System.* and Microsoft.*)
4. USED_CLASSES: What class/interface/type names are USED but NOT defined here?
5. GLOBAL_USINGS: What namespaces use global using statements?

Return ONLY valid JSON.

File: src/Services/AuthService.cs

```csharp
using MyApp.Models;
using MyApp.Repositories;

namespace MyApp.Services
{
    public class AuthService
    {
        private readonly IUserRepository _userRepo;
        public AuthService(IUserRepository userRepo) { _userRepo = userRepo; }
        public User Authenticate(string username) { return _userRepo.FindByName(username); }
    }
}
```"""

messages = [{"role": "user", "content": test_prompt}]
text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

FastLanguageModel.for_inference(model)
inputs = tokenizer(text, return_tensors="pt").to("cuda")
output = model.generate(**inputs, max_new_tokens=512, temperature=0.1, do_sample=True)
generated = tokenizer.decode(output[0][inputs["input_ids"].shape[-1]:], skip_special_tokens=True)
print(generated)

## Save

Save LoRA adapters + merged 16-bit model for vLLM.

In [ ]:
# Save LoRA adapters
model.save_pretrained(str(LORA_DIR))
tokenizer.save_pretrained(str(LORA_DIR))
print(f"LoRA adapters saved to: {LORA_DIR}")

In [ ]:
# Save merged 16-bit model for vLLM
model.save_pretrained_merged(
    str(OUTPUT_DIR),
    tokenizer,
    save_method="merged_16bit",
)
print(f"\nMerged 16-bit model saved to: {OUTPUT_DIR}")
print(f"\nServe with vLLM:")
print(f"  vllm serve {OUTPUT_DIR} --max-model-len 16384")

## Next steps

1. Start vLLM with the finetuned model:
   ```bash
   vllm serve ./finetuned_qwen3_dep --max-model-len 16384
   ```
2. Open `01_eval_base_models.ipynb`, add the finetuned model to `MODELS`:
   ```python
   MODELS["qwen3-4b-finetuned"] = {
       "hf_id": str(Path("./finetuned_qwen3_dep").resolve()),
       "workers": 4,
       "max_tokens": 1024,
       "extra_body": {"chat_template_kwargs": {"enable_thinking": False}},
   }
   CURRENT_MODEL = "qwen3-4b-finetuned"
   ```
3. Run all cells to evaluate finetuned vs Gemini